# 02 – EDA & Feature Engineering

This notebook loads the parsed log DataFrame, performs exploratory data analysis, engineers per-IP/per-hour features, and saves `data/processed/features.parquet`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PARSED = ROOT / "data" / "processed" / "parsed_logs.parquet"
df = pd.read_parquet(str(PARSED))
print(f"Loaded {len(df):,} rows.")
df.head(3)


In [ ]:
# ── EDA: request volume over time ────────────────────────────────────────────
df["hour"] = df["timestamp"].dt.floor("h")
hourly = df.groupby("hour").size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(hourly["hour"], hourly["count"], width=0.03)
ax.set_xlabel("Hour"); ax.set_ylabel("Requests"); ax.set_title("Requests per Hour")
plt.tight_layout(); plt.show()


In [ ]:
# ── EDA: status code distribution ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3))
df["status_code"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xlabel("Status Code"); ax.set_ylabel("Count"); ax.set_title("HTTP Status Distribution")
plt.tight_layout(); plt.show()


In [ ]:
# ── EDA: top IPs ─────────────────────────────────────────────────────────────
top_ips = df["ip_address"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(7, 4))
top_ips.plot(kind="barh", ax=ax)
ax.set_xlabel("Request Count"); ax.set_title("Top 15 IP Addresses")
plt.tight_layout(); plt.show()


In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
from pipeline.feature_engineering import engineer_features

feat_df = engineer_features(df)
print(f"Feature matrix: {feat_df.shape}")
feat_df.head()


In [ ]:
# ── Feature correlation heatmap ───────────────────────────────────────────────
FEAT_COLS = [
    "requests_per_hour", "error_rate", "unique_endpoints",
    "avg_bytes_sent", "post_ratio", "is_off_hours", "is_weekend", "has_scanner_ua"
]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(feat_df[FEAT_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout(); plt.show()


In [ ]:
# ── Save features ─────────────────────────────────────────────────────────────
OUT = ROOT / "data" / "processed" / "features.parquet"
feat_df.to_parquet(str(OUT), index=False)
print(f"Saved → {OUT}  ({len(feat_df):,} rows)")
feat_df[FEAT_COLS].describe().round(3)
